# NL2Cypher — cloud experiment runner (Gemini / OpenAI)

Runs Settings 1–3 (`DIRECT_QA_BASELINE`, `DIRECT_QA_GROUNDED`, `CYPHER_SOFT`) against
a remote Neo4j server using a Gemini or OpenAI engine. `CYPHER_STRICT` stays on the
local notebook because Outlines requires a GPU.

**Inputs**
1. A server-produced `bundle_<model>.json` (from `scripts/build_bundle.py`).
2. Bolt access to the Neo4j host that owns the graph.
3. `LLM_API_KEY` for the chosen provider.

A CPU Colab runtime is enough — no torch installed here.

## 1. Clone the repo and install dependencies

In [ ]:
%%bash
set -e
if [ ! -d ec3_nl2cypher ]; then
  git clone --depth 1 https://github.com/YOUR_FORK/nl2cypher-on-steroids.git repo
  mv repo/ec3_nl2cypher .
fi
cd ec3_nl2cypher
pip install -q -r requirements-base.txt
# Cloud runs only need the Gemini/OpenAI SDKs — skip torch/outlines.
pip install -q google-generativeai openai

In [ ]:
import os, sys
sys.path.insert(0, 'ec3_nl2cypher')

# Server connection details. Replace with your Bolt endpoint.
os.environ['NEO4J_URI']       = 'bolt://YOUR_SERVER:7687'
os.environ['NEO4J_USER']      = 'neo4j'
os.environ['NEO4J_PASSWORD']  = 'REPLACE_ME'

# LLM provider. Switch to 'openai' + OPENAI_API_KEY if preferred.
os.environ['LLM_PROVIDER']    = 'gemini'
os.environ['LLM_MODEL_NAME']  = 'gemini-1.5-flash'
os.environ['LLM_API_KEY']     = 'REPLACE_ME'

## 2. Load the bundle produced server-side

Upload `bundle_<model>.json` to Colab (drag into the file pane, or mount Drive).

In [ ]:
import json, pathlib

BUNDLE_PATH = pathlib.Path('bundle_barcelona.json')
bundle = json.loads(BUNDLE_PATH.read_text())
print('bundle keys:', list(bundle))
print('vocab entities:', len(bundle['combined_vocabulary']['entities']))
print('model_dump elements:', len(bundle['model_dump']))
print('graph stats labels:', len(bundle['graph_stats'].get('labels', {})))

In [ ]:
# Write model_dump to a file so the ExperimentRunner can consume it via --model-dump.
MODEL_DUMP_PATH = pathlib.Path('model_dump.json')
MODEL_DUMP_PATH.write_text(json.dumps(bundle['model_dump']))
print(f'wrote {MODEL_DUMP_PATH} ({MODEL_DUMP_PATH.stat().st_size // 1024} KB)')

## 3. Run the experiment

The runner talks to Neo4j over Bolt for Cypher settings and feeds `model_dump.json`
into the LLM prompt for Direct QA settings.

In [ ]:
from pathlib import Path
from src.config import ExperimentSetting
from src.eval.runner import ExperimentConfig, ExperimentRunner

config = ExperimentConfig(
    name='cloud_run',
    test_set_path=Path('ec3_nl2cypher/data/test_set.csv'),
    output_dir=Path('results'),
    model_dump_path=MODEL_DUMP_PATH,
    cloud_direct=False,
    settings=[
        ExperimentSetting.DIRECT_QA_BASELINE,
        ExperimentSetting.DIRECT_QA_GROUNDED,
        ExperimentSetting.CYPHER_SOFT,
    ],
)

runner = ExperimentRunner(config=config)
runner.run_comparison(
    test_set_path=config.test_set_path,
    output_dir=config.output_dir,
    settings=config.settings,
)

## 4. Inspect results

CSV + JSON summaries land in `results/`. Copy to Drive or download with the file browser.

In [ ]:
import pandas as pd, glob
for path in sorted(glob.glob('results/*_summary.json')):
    print(path)
    print(json.dumps(json.loads(open(path).read()), indent=2))
    print()